# 🗄️ Notebook 0: Exploratory Analysis & Dataset Limitations Study

**Author:** Gabriella Marín  
**Project:** Water Quality Normative Compliance Analysis & Potability Prediction

---

## 📌 Initial Project Objective

The initial goal of this project was to:
1. Build a SQLite database (`water_quality_analysis.db`) for water quality assessment
2. Apply three regulatory frameworks (WHO, EPA, Colombia Resolution 2115/2007)
3. Generate independent potability classifications per standard
4. Perform comparative and predictive analysis based on regulatory compliance

---

## 🔍 Exploratory Findings & Limitations

During the exploratory and normative calculation stage, the following limitations were identified:

- The dataset contains a high proportion of missing values in key parameters.
- Most physicochemical parameters systematically exceed potable ranges, resulting in **zero samples classified as potable** under regulatory thresholds.
- The absence of essential microbiological parameters (e.g., *E. coli*) prevents realistic potability determination.
- The lack of contextual and monitoring information limits the applicability of regulatory standards.

As a result, continuing with normative comparison or predictive modeling using this dataset would lead to **non-representative and technically invalid conclusions**.

---

## 📌 Scope Adjustment

This notebook is therefore retained as an **exploratory analysis and dataset suitability assessment**.

The full normative compliance analysis and predictive modeling are conducted using alternative datasets that provide:
- regulatory-aligned parameters,
- sufficient data completeness,
- and realistic applicability for water quality assessment.

---

*Identifying dataset limitations and redefining analytical scope is a critical step in applied environmental data science.*

## 1. Initial Setup and Data Loading

In [16]:
# Imports
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Paths configuration
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
DB_PATH = DATA_DIR / 'water_quality_analysis.db'

# Load original dataset
df = pd.read_csv(RAW_DIR / 'water_potability.csv')

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

✅ Dataset loaded: 3276 rows × 10 columns


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0


## 2. Normative Limits Definition

Regulatory standards for water quality from three organizations:
- **WHO**: World Health Organization
- **EPA**: Environmental Protection Agency (USA)
- **Colombia**: Resolución 2115 de 2007

In [6]:
# Normative limits for each organization
normative_limits = {
    'WHO': {
        'ph_min': 6.5, 'ph_max': 8.5,
        'turbidity_max': 5.0,
        'chloramines_max': 3.0,
        'sulfate_max': 500.0,
        'solids_max': 1000.0,
        'trihalomethanes_max': 100.0,
        'hardness_max': None  # WHO doesn't specify hardness limit
    },
    'EPA': {
        'ph_min': 6.5, 'ph_max': 8.5,
        'turbidity_max': 5.0,
        'chloramines_max': 4.0,
        'sulfate_max': 250.0,  # More restrictive than WHO
        'solids_max': 500.0,   # More restrictive than WHO
        'trihalomethanes_max': 80.0,  # More restrictive than WHO
        'hardness_max': None
    },
    'Colombia': {
        'ph_min': 6.5, 'ph_max': 9.0,  # More permissive pH
        'turbidity_max': 2.0,  # Stricter turbidity (most restrictive)
        'chloramines_max': 3.0,
        'sulfate_max': 250.0,
        'solids_max': 1000.0,
        'trihalomethanes_max': 200.0,  # More permissive THMs
        'hardness_max': 300.0  # Only Colombia specifies hardness
    }
}

print("✅ Normative limits defined")
print(f"Organizations: {list(normative_limits.keys())}")

✅ Normative limits defined
Organizations: ['WHO', 'EPA', 'Colombia']


## 3. Potability Classification Functions

Functions to evaluate if a water sample is potable according to each normative framework.

**Conservative approach:** Samples with missing values in critical parameters are classified as non-potable.

In [7]:
def check_potability_who(row):
    """
    Check if water sample is potable according to WHO standards.
    Returns: 1 (potable) or 0 (not potable)
    """
    # Handle missing values (conservative approach)
    if pd.isna(row['ph']) or pd.isna(row['Turbidity']) or pd.isna(row['Chloramines']) or \
       pd.isna(row['Sulfate']) or pd.isna(row['Solids']) or pd.isna(row['Trihalomethanes']):
        return 0
    
    # Check each parameter against WHO limits
    ph_ok = normative_limits['WHO']['ph_min'] <= row['ph'] <= normative_limits['WHO']['ph_max']
    turbidity_ok = row['Turbidity'] <= normative_limits['WHO']['turbidity_max']
    chloramines_ok = row['Chloramines'] <= normative_limits['WHO']['chloramines_max']
    sulfate_ok = row['Sulfate'] <= normative_limits['WHO']['sulfate_max']
    solids_ok = row['Solids'] <= normative_limits['WHO']['solids_max']
    thms_ok = row['Trihalomethanes'] <= normative_limits['WHO']['trihalomethanes_max']
    
    # All parameters must be within limits
    return 1 if all([ph_ok, turbidity_ok, chloramines_ok, sulfate_ok, solids_ok, thms_ok]) else 0


def check_potability_epa(row):
    """
    Check if water sample is potable according to EPA standards.
    Returns: 1 (potable) or 0 (not potable)
    """
    # Handle missing values
    if pd.isna(row['ph']) or pd.isna(row['Turbidity']) or pd.isna(row['Chloramines']) or \
       pd.isna(row['Sulfate']) or pd.isna(row['Solids']) or pd.isna(row['Trihalomethanes']):
        return 0
    
    # Check each parameter against EPA limits
    ph_ok = normative_limits['EPA']['ph_min'] <= row['ph'] <= normative_limits['EPA']['ph_max']
    turbidity_ok = row['Turbidity'] <= normative_limits['EPA']['turbidity_max']
    chloramines_ok = row['Chloramines'] <= normative_limits['EPA']['chloramines_max']
    sulfate_ok = row['Sulfate'] <= normative_limits['EPA']['sulfate_max']
    solids_ok = row['Solids'] <= normative_limits['EPA']['solids_max']
    thms_ok = row['Trihalomethanes'] <= normative_limits['EPA']['trihalomethanes_max']
    
    return 1 if all([ph_ok, turbidity_ok, chloramines_ok, sulfate_ok, solids_ok, thms_ok]) else 0


def check_potability_colombia(row):
    """
    Check if water sample is potable according to Colombia Resolución 2115.
    Returns: 1 (potable) or 0 (not potable)
    """
    # Handle missing values (including Hardness for Colombia)
    if pd.isna(row['ph']) or pd.isna(row['Turbidity']) or pd.isna(row['Chloramines']) or \
       pd.isna(row['Sulfate']) or pd.isna(row['Solids']) or pd.isna(row['Trihalomethanes']) or \
       pd.isna(row['Hardness']):
        return 0
    
    # Check each parameter against Colombia limits
    ph_ok = normative_limits['Colombia']['ph_min'] <= row['ph'] <= normative_limits['Colombia']['ph_max']
    turbidity_ok = row['Turbidity'] <= normative_limits['Colombia']['turbidity_max']
    chloramines_ok = row['Chloramines'] <= normative_limits['Colombia']['chloramines_max']
    sulfate_ok = row['Sulfate'] <= normative_limits['Colombia']['sulfate_max']
    solids_ok = row['Solids'] <= normative_limits['Colombia']['solids_max']
    thms_ok = row['Trihalomethanes'] <= normative_limits['Colombia']['trihalomethanes_max']
    hardness_ok = row['Hardness'] <= normative_limits['Colombia']['hardness_max']
    
    return 1 if all([ph_ok, turbidity_ok, chloramines_ok, sulfate_ok, solids_ok, thms_ok, hardness_ok]) else 0


print("✅ Classification functions created")

✅ Classification functions created


In [15]:
# Test functions with first sample
test_sample = df.iloc[3]  # Get first row

print("Testing functions with first sample:")
print(f"Sample pH: {test_sample['ph']}, Turbidity: {test_sample['Turbidity']}")
print(f"\nWHO classification: {check_potability_who(test_sample)}")
print(f"EPA classification: {check_potability_epa(test_sample)}")
print(f"Colombia classification: {check_potability_colombia(test_sample)}")

Testing functions with first sample:
Sample pH: 8.316765884214679, Turbidity: 4.628770536837084

WHO classification: 0
EPA classification: 0
Colombia classification: 0


## 4. Calculate Potability Classifications

Apply classification functions to all samples to generate three new potability columns (WHO, EPA, Colombia).

In [17]:
# Apply classification functions to all rows
df['Potability_WHO'] = df.apply(check_potability_who, axis=1)
df['Potability_EPA'] = df.apply(check_potability_epa, axis=1)
df['Potability_Colombia'] = df.apply(check_potability_colombia, axis=1)

# Summary statistics
print("✅ Potability classifications calculated")
print(f"\nPotability Summary:")
print(f"  Original dataset:  {df['Potability'].sum():4.0f} potable ({df['Potability'].sum()/len(df)*100:5.1f}%)")
print(f"  WHO standard:      {df['Potability_WHO'].sum():4.0f} potable ({df['Potability_WHO'].sum()/len(df)*100:5.1f}%)")
print(f"  EPA standard:      {df['Potability_EPA'].sum():4.0f} potable ({df['Potability_EPA'].sum()/len(df)*100:5.1f}%)")
print(f"  Colombia standard: {df['Potability_Colombia'].sum():4.0f} potable ({df['Potability_Colombia'].sum()/len(df)*100:5.1f}%)")

# Display sample with new columns
print("\n🔍 Sample with new classifications:")
df[['ph', 'Turbidity', 'Hardness', 'Potability', 'Potability_WHO', 'Potability_EPA', 'Potability_Colombia']].head(5)

✅ Potability classifications calculated

Potability Summary:
  Original dataset:  1278 potable ( 39.0%)
  WHO standard:         0 potable (  0.0%)
  EPA standard:         0 potable (  0.0%)
  Colombia standard:    0 potable (  0.0%)

🔍 Sample with new classifications:


,ph,Turbidity,Hardness,Potability,Potability_WHO,Potability_EPA,Potability_Colombia
0,NaN,2.963135,204.890455,0,0,0,0
1,3.716080,4.500656,129.422921,0,0,0,0
2,8.099124,3.055934,224.236259,0,0,0,0
3,8.316766,4.628771,214.373394,0,0,0,0
4,9.092223,4.075075,181.101509,0,0,0,0


In [18]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nRows with ANY missing value: {df.isnull().any(axis=1).sum()} ({df.isnull().any(axis=1).sum()/len(df)*100:.1f}%)")

Missing values per column:
ph                     491
Hardness                 0
Solids                   0
Chloramines              0
Sulfate                781
Conductivity             0
Organic_carbon           0
Trihalomethanes        162
Turbidity                0
Potability               0
Potability_WHO           0
Potability_EPA           0
Potability_Colombia      0
dtype: int64

Rows with ANY missing value: 1265 (38.6%)
